# 05 · CICIoT2023 (`IoT23`) — Phase 2 Data understanding · Phase 3 Data preprocessing
**Owner:** Yassine Mtibaa · **Project:** Cross-Network Intrusion Analysis (CRISP-DM)

Structure required by `notebooks/README.md`, following the course lab:

| Phase 2 — diagnosis only (lab Part 1) | Phase 3 — apply the plan (lab Part 2) |
|---|---|
| 2.1 Initial inspection | 3.1 Missing and infinite values |
| 2.2 Missing values | 3.2 Data type conversion |
| 2.3 Data type issues | 3.3 Standardise column names and labels (`map_labels`) |
| 2.4 Outlier detection | 3.4 Outlier treatment |
| 2.5 Consistency | 3.5 Duplicates and identifier columns |
| 2.6 Class distribution | 3.6 Validation (`validate`) and quality report |
| 2.7 Summary: problem → planned solution | 3.7 Stratified sampling (`sample_per_label`) and save |

**Data:** the `MERGED_CSV` release of CICIoT2023 (63 files `Merged01.csv … Merged63.csv`, 9.3 GB) in `data/raw/CICIoT2023/`.
Files are read **in parallel, one per worker**; every count in phase 2 is computed on **all rows**, plots use random samples.
Phase 3 needs about 16 GB of RAM (the cleaned data is held in memory once to remove duplicates across files).

In [ ]:
import sys; sys.path.append("..")
from src.config import *

import time, pickle, multiprocessing as mp
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 60)

SOURCE = "IoT23"
LABEL_COL = "Label"          # the MERGED_CSV release uses 'Label' (config.CICIOT_LABEL_COL says 'label', see decision log)
DATA = RAW_DIR / DATASETS[SOURCE]["file"]
all_csv = sorted(DATA.rglob("*.csv"))
# The CIC download contains the same data twice (MERGED_CSV/ and CSV/<attack>/): read only one copy.
merged = [p for p in all_csv if "merged" in p.name.lower() or "merged_csv" in str(p).lower()]
FILES = merged if merged else all_csv
CACHE = PROCESSED_DIR / "IoT23_phase2_profile.pkl"   # git-ignored cache of the full pass
print(f"{len(FILES)} files, {sum(f.stat().st_size for f in FILES)/1e9:.2f} GB in {DATA}")

---
# Phase 2 — Data understanding (diagnosis only, nothing is changed)
## 2.1 Initial inspection
### One file: structure, types, first rows

In [ ]:
first = pd.read_csv(FILES[0])
print(first.shape)
first.info()
first.head()

### Full pass over all files
`profile_file` reads one file and returns only small summaries: counts per label, NaN/inf/negative counts per column, min/max/sum per column, the result of each logical rule, per-class anomaly counts, row hashes (to find duplicates across files) and two random samples. Ten files are processed at the same time.

The **logical rules** test relations that must hold if the columns mean what we think they mean. A rule with **0 violations** proves that one column is computed from others (redundant); a rule with violations reveals inconsistent values.

In [ ]:
SHARE = ["fin_flag_number", "syn_flag_number", "rst_flag_number", "psh_flag_number", "ack_flag_number", "ece_flag_number",
         "cwr_flag_number", "HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"]
COUNTS = ["ack_count", "syn_count", "fin_count", "rst_count"]
PAIRS = dict(zip(COUNTS, ["ack_flag_number", "syn_flag_number", "fin_flag_number", "rst_flag_number"]))
WHOLE = COUNTS + ["Protocol Type", "Tot sum", "Min", "Max", "Number"]

RULES = {
    "Min > Max":                                lambda d: d["Min"] > d["Max"],
    "AVG outside [Min, Max]":                   lambda d: (d["AVG"] < d["Min"] - 1e-6) | (d["AVG"] > d["Max"] + 1e-6),
    "Std < 0 or Variance < 0":                  lambda d: (d["Std"] < 0) | (d["Variance"] < 0),
    "Time_To_Live outside [0, 255]":            lambda d: (d["Time_To_Live"] < 0) | (d["Time_To_Live"] > 255),
    "Number outside [1, 100]":                  lambda d: (d["Number"] < 1) | (d["Number"] > 100),
    "a *_count > Number":                       lambda d: d[COUNTS].max(axis=1) > d["Number"],
    "a share column outside [0, 1]":            lambda d: ((d[SHARE] < 0) | (d[SHARE] > 1)).any(axis=1),
    "Header_Length > 60 (max TCP header)":      lambda d: d["Header_Length"] > 60,
    "IAT < 0":                                  lambda d: d["IAT"] < 0,
    "Tot size != AVG":                          lambda d: ~np.isclose(d["Tot size"], d["AVG"], rtol=0, atol=1e-6),
    "IPv != LLC":                               lambda d: ~np.isclose(d["IPv"], d["LLC"], rtol=0, atol=1e-9),
    "Variance != Std^2":                        lambda d: ~np.isclose(d["Variance"], d["Std"] ** 2, rtol=0.02, atol=0.02),
    "Tot sum != AVG x Number":                  lambda d: ~np.isclose(d["Tot sum"], d["AVG"] * d["Number"], rtol=0.01, atol=1),
    "x_count != x_flag_number x Number":        lambda d: np.column_stack([~np.isclose(d[c], d[f] * d["Number"], rtol=0, atol=1.01) for c, f in PAIRS.items()]).any(axis=1),
    "TCP share > 0.5 but Protocol Type != 6":   lambda d: (d["TCP"] > 0.5) & (d["Protocol Type"] != 6),
    "UDP share > 0.5 but Protocol Type != 17":  lambda d: (d["UDP"] > 0.5) & (d["Protocol Type"] != 17),
    "ICMP share > 0.5 but Protocol Type != 1":  lambda d: (d["ICMP"] > 0.5) & (d["Protocol Type"] != 1),
}
ANOMALIES = {   # counted per class, to understand WHERE each anomaly happens
    "Header_Length = 0":    lambda d: d["Header_Length"] == 0,
    "Time_To_Live = 0":     lambda d: d["Time_To_Live"] == 0,
    "Max > 1514 bytes":     lambda d: d["Max"] > 1514,
    "Rate = inf":           lambda d: np.isinf(d["Rate"]),
    "IAT > 1 hour":         lambda d: d["IAT"] > 3600,
    "Protocol Type = 0":    lambda d: d["Protocol Type"] == 0,
    "Number < 100":         lambda d: d["Number"] < 100,
    "Std = NaN":            lambda d: d["Std"].isna(),
}

def profile_file(args):
    path, seed = args
    raw = pd.read_csv(path, low_memory=False)
    d = raw.rename(columns={c: c.strip() for c in raw.columns})
    feats = [c for c in d.columns if c != LABEL_COL]
    num = d[feats].astype("float64")
    fin = num.replace([np.inf, -np.inf], np.nan)
    lab = d[LABEL_COL]
    r = {"file": path.name, "rows": len(d), "cols": tuple(d.columns), "dtypes": {c: str(t) for c, t in raw.dtypes.items()},
         "labels": lab.value_counts(dropna=False).to_dict(),
         "nan": d.isna().sum().to_dict(), "inf": np.isinf(num).sum().to_dict(), "neg": (fin < 0).sum().to_dict(),
         "min": fin.min().to_dict(), "max": fin.max().to_dict(), "sum": fin.sum().to_dict(), "sumsq": (fin ** 2).sum().to_dict(),
         "n": fin.count().to_dict(), "nonint": {c: int((fin[c] % 1 != 0).sum()) for c in WHOLE},
         "proto": fin["Protocol Type"].value_counts().to_dict(),
         "broken": [(path.name, int(i), int(k)) for i, k in d.isna().sum(axis=1).items() if k >= 5],
         "std_nan_number": d.loc[d["Std"].isna() & (d.isna().sum(axis=1) < 5), "Number"].value_counts().to_dict()}
    r["rules"] = {k: int(np.asarray(pd.Series(f(fin)).fillna(False), dtype=bool).sum()) for k, f in RULES.items()}
    r["anom"] = {k: pd.Series(np.asarray(f(d.assign(Rate=num["Rate"])), dtype=bool)).groupby(lab.values).sum().to_dict() for k, f in ANOMALIES.items()}
    r["h_feat"] = pd.util.hash_pandas_object(d[feats], index=False).values
    lh = pd.util.hash_pandas_object(lab.astype(str), index=False).values       # labels as 64-bit codes (saves memory)
    r["lab_arr"] = lh; r["lab_map"] = dict(zip(lh, lab.astype(str).values))
    rng = np.random.default_rng(seed); key = rng.random(len(d))
    r["uniform"] = d.iloc[np.argsort(key)[:12_000]]                                   # ~750k rows over 63 files
    r["by_class"] = d.assign(_k=key)[lambda x: x.groupby(LABEL_COL)["_k"].rank(method="first") <= 400].drop(columns="_k")
    return r

if CACHE.exists():
    P = pickle.load(open(CACHE, "rb"))
else:
    t0 = time.time()
    with mp.get_context("fork").Pool(10) as pool:
        res = pool.map(profile_file, [(f, RANDOM_STATE + i) for i, f in enumerate(FILES)])
    print(f"full pass: {time.time() - t0:.0f}s")
    tot = lambda key: pd.Series(sum((Counter(r[key]) for r in res), Counter()))
    H = pd.DataFrame({"h": np.concatenate([r["h_feat"] for r in res]), "lab": np.concatenate([r["lab_arr"] for r in res])})
    LMAP = {k: v for r in res for k, v in r["lab_map"].items()}
    n_lab = H.groupby("h")["lab"].nunique()
    P = {"files": len(res), "rows": sum(r["rows"] for r in res), "layouts": len({r["cols"] for r in res}),
         "rows_per_file": pd.Series({r["file"]: r["rows"] for r in res}),
         "dtype_variants": {c: sorted({r["dtypes"].get(c) for r in res}) for c in res[0]["dtypes"]},
         "labels": tot("labels"), "nan": tot("nan"), "inf": tot("inf"), "neg": tot("neg"), "nonint": tot("nonint"), "proto": tot("proto"),
         "min": pd.DataFrame([r["min"] for r in res]).min(), "max": pd.DataFrame([r["max"] for r in res]).max(),
         "mean": tot("sum") / tot("n"), "std": np.sqrt((tot("sumsq") / tot("n") - (tot("sum") / tot("n")) ** 2).clip(lower=0)),
         "rules": tot("rules"), "broken": [b for r in res for b in r["broken"]], "std_nan_number": tot("std_nan_number"),
         "anom": pd.DataFrame({k: sum((Counter(r["anom"][k]) for r in res), Counter()) for k in ANOMALIES}).fillna(0).astype(int),
         "dup_rows": int(H.duplicated().sum()),                      # same features AND same label
         "dup_features": int(H["h"].duplicated().sum()),             # same features, any label
         "conflict_vectors": int((n_lab > 1).sum()),
         "conflict_rows_by_label": H[H["h"].isin(n_lab[n_lab > 1].index)]["lab"].map(LMAP).value_counts(),
         "dup_rows_by_label": H[H.duplicated()]["lab"].map(LMAP).value_counts(),
         "uniform": pd.concat([r["uniform"] for r in res], ignore_index=True),
         "by_class": pd.concat([r["by_class"] for r in res], ignore_index=True)}
    del res, H
    pickle.dump(P, open(CACHE, "wb"))
R = P["rows"]
print(f"{R:,} rows | {P['files']} files | {P['layouts']} column layout(s)")

In [ ]:
labels = P["labels"]; labels = labels[labels.index.notna()].sort_values(ascending=False).astype(int)   # used by later cells

### Rows and columns: official vs loaded

In [ ]:
official = pd.DataFrame({
    "docs/datasets.md": ["~47 million", "45 features"],
    "UNB website (2023 release)": ["46,686,579", "46 features + label"],
    "Loaded (MERGED_CSV release)": [f"{R:,}", f"{first.shape[1] - 1} features + Label"],
}, index=["Rows", "Columns"])
official

**Interpretation.** The files we downloaded are a **newer release** than the one described on the UNB page and in `docs/datasets.md`: 63 merged files instead of 169, and **40 columns instead of 47**. Compared with the original release, `Duration` was renamed `Time_To_Live`, `IGMP` was added, and `flow_duration`, `Srate`, `Drate`, `urg_count`, `Magnitue`, `Radius`, `Covariance` and `Weight` were removed. The row count is also slightly lower. All numbers in this project therefore come from the loaded files, not from the documentation.

**Granularity — what one row is.** A row is **one window of up to 100 packets** (`Number` = packets in the window, median 100). Every feature is a summary over that window: an average, a share between 0 and 1, a count, or a size statistic. There is **no identifier column** (no IP, port, device or timestamp).

### Columns, types and units (grouped by role)

In [ ]:
groups = {
    "Identifiers": [],
    "Ports / protocol": ["Protocol Type", "Header_Length", "Time_To_Live"],
    "Protocol shares (0-1)": ["HTTP", "HTTPS", "DNS", "Telnet", "SMTP", "SSH", "IRC", "TCP", "UDP", "DHCP", "ARP", "ICMP", "IGMP", "IPv", "LLC"],
    "Volume (bytes, packets)": ["Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "Variance", "Number"],
    "Time (rate, IAT)": ["Rate", "IAT"],
    "TCP flags": SHARE[:7] + COUNTS,
    "Labels": [LABEL_COL],
}
UNITS = {"Protocol Type": "IANA code (1 ICMP, 2 IGMP, 6 TCP, 17 UDP, 47 GRE, 0 other)", "Header_Length": "bytes (window average)",
         "Time_To_Live": "hops (window average)", "Tot sum": "bytes", "Min": "bytes", "Max": "bytes", "AVG": "bytes", "Std": "bytes",
         "Tot size": "bytes", "Variance": "bytes²", "Number": "packets in window", "Rate": "packets / s", "IAT": "seconds",
         LABEL_COL: "34 classes (text)"}
rows = []
for g, cols in groups.items():
    for c in cols:
        rows.append({"group": g, "column": c, "dtype": str(first[c].dtype),
                     "unit": UNITS.get(c, "share of packets in window (0-1)" if c in SHARE else "packets with this flag (0-100)" if c in COUNTS else ""),
                     "min": P["min"].get(c), "max": P["max"].get(c), "mean": P["mean"].get(c)})
dtypes_table = pd.DataFrame(rows)
assert set(dtypes_table["column"]) == set(first.columns), "every column must appear in exactly one group"
dtypes_table.to_csv(TABLE_DIR / "IoT23_phase2_columns.csv", index=False)
dtypes_table.round(4)

**First visible issues** (each one is measured in the sections below):
* No identifier, port or timestamp column — rows cannot be traced to a device or ordered in time.
* Three column names contain spaces (`Protocol Type`, `Tot sum`, `Tot size`).
* `Protocol Type` is a code stored as a number; the `*_flag_number` and protocol columns are shares (0–1) rather than 0/1 flags.
* Several columns look like copies of each other (`AVG` / `Tot size`, `IPv` / `LLC`) and several are almost always 0.
* Labels are written in UPPERCASE, unlike the documentation and `config.TAXONOMY`.

## 2.2 Missing values analysis
`missing_report` (shared function) on one file, then the same measure on all 45 M rows (missing and infinite, count and %).

In [ ]:
display(missing_report(first))

In [ ]:
mr = pd.DataFrame({"missing": P["nan"], "infinite": P["inf"]}).fillna(0).astype(int)
mr = mr[(mr["missing"] > 0) | (mr["infinite"] > 0)]
mr["missing_pct"] = (100 * mr["missing"] / R).round(5); mr["infinite_pct"] = (100 * mr["infinite"] / R).round(5)
mr = mr.sort_values(["missing", "infinite"], ascending=False)
mr.to_csv(TABLE_DIR / "IoT23_phase2_missing.csv")

fig, ax = plt.subplots(figsize=(12, 4))
mr[["missing", "infinite"]].plot.bar(ax=ax, color=["#5B6B8C", "#E8871E"], logy=True)
ax.set_title(f"Missing and infinite values per column (all {R/1e6:.1f} M rows, log scale)"); ax.set_ylabel("cells")
plt.tight_layout(); plt.savefig(FIG_DIR / "IoT23_missing_values.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"truncated lines (>= 5 empty cells): {len(P['broken'])} ->", P["broken"])
print("Std = NaN outside those lines, by Number:", P["std_nan_number"].astype(int).to_dict())
mr

**Interpretation.**
* **7 truncated lines** (one in each of `Merged42/44/46/48/50/51/52`) have 12–36 empty cells, and 9 rows have no label: these cause almost all NaN values and the int/float dtype differences.
* `Std` and `Variance` are also NaN in **670 rows, all with `Number = 1`**: the standard deviation of a single packet is undefined. This is *structural* missingness (MNAR), not random.
* `Rate` is **infinite in 991 rows** (division by zero when the window lasts 0 s), spread over many classes, benign included.
* `IAT` is negative in 2 rows, which is physically impossible.

**Planned strategy per affected column**

| Column(s) | Cause | Rows | Strategy (phase 3) | Why |
|---|---|---|---|---|
| ~30 columns, `Label` | 7 truncated lines + 9 rows without label | 16 | **drop the rows** | broken records; no value can be trusted |
| `Std`, `Variance` | window with a single packet (`Number` = 1) | 670 | **impute 0** | the spread of one value is 0; structural, not random — dropping would remove every single-packet window |
| `Rate` | division by zero (window of 0 s) | 991 | inf → NaN → **drop the rows** | 0.002 % of rows, spread over many classes, no reliable value to impute |
| `IAT` (negative) | impossible value | 2 | set to NaN → **drop** | a negative time between packets cannot exist (treated in 2.4 / 3.4) |

## 2.3 Data type issues

In [ ]:
print("columns whose dtype differs between files:", {c: v for c, v in P["dtype_variants"].items() if len(v) > 1})
print("Protocol Type values (rows):", P["proto"].astype(int).to_dict())
print("Number of labels:", int(P["labels"].index.notna().sum()))

In [ ]:
type_issues = pd.DataFrame([
    ("numbers stored as text", "none", "—", "—", "no action (all 39 features parse as numbers)"),
    ("timestamps stored as integers", "none (no timestamp column)", "—", "—", "no action; limitation: no time-based analysis"),
    ("integers read as float in 7 files", "ack/syn/fin/rst_count, Tot sum, Min, Max, Number", "float64 (some files)", "integer",
     "drop the truncated lines, check every value is whole, cast to integer"),
    ("code stored as number", "Protocol Type", "int64", "category", "cast to category (values 0, 1, 2, 6, 17, 47); one-hot encoding is phase 5"),
    ("shares named like flags", "*_flag_number (7)", "float64", "float64 (share 0-1)", "keep; documented in the feature dictionary"),
    ("column names with spaces", "Protocol Type, Tot sum, Tot size", "—", "—", "replace spaces with '_'"),
    ("label spellings differ from TAXONOMY", "Label (33 of 34)", "text, UPPERCASE", "category", "map_labels() -> attack_label, attack_family"),
], columns=["issue", "column(s)", "current type", "correct type", "planned fix"])
type_issues

**Interpretation.**
* All 39 features are numeric; `Label` is text. There are **no identifier, port or timestamp columns** at all.
* `Protocol Type` is stored as an integer but is a **categorical code** (only 6 distinct values).
* The 7 `*_flag_number` columns and the 15 protocol columns are **shares between 0 and 1** (e.g. 0.99), not 0/1 booleans as their names suggest, because every row summarises a window of packets.
* Eight integer columns are read as `float64` in a few files: those files contain truncated lines with empty cells (see 2.4).

### Label spellings

In [ ]:
spell = pd.DataFrame({"label in the files": labels.index,
                      "matching key in config.TAXONOMY": [next(k for k in TAXONOMY if k.lower().replace("-", "").replace("_", "").replace(" ", "") ==
                                                               l.lower().replace("-", "").replace("_", "").replace(" ", "")) for l in labels.index]})
print("labels written differently from TAXONOMY (only the case/separators differ; map_labels normalises them):",
      int((spell.iloc[:, 0] != spell.iloc[:, 1]).sum()), "of", len(spell))
spell.head(8)

## 2.4 Outlier detection
Quantiles need every value of a column, so the IQR and z-score counts use the **uniform random sample** (~750k rows drawn evenly from the 63 files);
physical-limit checks and minima/maxima use **all rows**. The 22 share columns are bounded in [0, 1] and are not tested.
* **IQR rule:** below Q1 − 1.5·IQR or above Q3 + 1.5·IQR (mild); 3·IQR (extreme).
* **z-score:** |z| > 3. Network features are very skewed, so the IQR rule is also applied on log10(1 + x).

In [ ]:
MEASURES = ["Header_Length", "Time_To_Live", "Rate", "Tot sum", "Min", "Max", "AVG", "Std", "Tot size", "Variance",
            "IAT", "Number", "ack_count", "syn_count", "fin_count", "rst_count"]
U = P["uniform"][MEASURES].replace([np.inf, -np.inf], np.nan)
rows = []
for c in MEASURES:
    x = U[c].dropna(); q1, q3 = x.quantile([.25, .75]); iqr = q3 - q1
    lx = np.log10(x.clip(lower=0) + 1); l1, l3 = lx.quantile([.25, .75]); liqr = l3 - l1
    z = (x - x.mean()) / x.std() if x.std() > 0 else x * 0
    rows.append({"column": c, "Q1": q1, "median": x.median(), "Q3": q3, "IQR": iqr,
                 "mild % (1.5 IQR)": 100 * ((x < q1 - 1.5 * iqr) | (x > q3 + 1.5 * iqr)).mean(),
                 "extreme % (3 IQR)": 100 * ((x < q1 - 3 * iqr) | (x > q3 + 3 * iqr)).mean(),
                 "|z| > 3 %": 100 * (z.abs() > 3).mean(),
                 "IQR on log scale %": 100 * ((lx < l1 - 1.5 * liqr) | (lx > l3 + 1.5 * liqr)).mean(),
                 "min (all rows)": P["min"][c], "max (all rows)": P["max"][c], "skewness": x.skew()})
outliers = pd.DataFrame(rows).set_index("column").round(3)
outliers.to_csv(TABLE_DIR / "IoT23_phase2_outliers.csv")
outliers.sort_values("mild % (1.5 IQR)", ascending=False)

**Reading the table.** When the IQR is 0 (e.g. `Number`, `Min`, `Time_To_Live`, where most windows share the same value), *every* other value counts as an outlier:
the IQR rule over-flags columns dominated by one value. The z-score flags far fewer rows because the extreme values inflate the standard deviation itself.
On a log scale most columns have few outliers left — the "outliers" are the long right tail of skewed traffic, not isolated errors.

In [ ]:
BOX = ["Rate", "IAT", "AVG", "Std", "Max", "Header_Length", "Time_To_Live", "Tot sum"]
bc = map_labels(P["by_class"], LABEL_COL, SOURCE)
order = bc.groupby("attack_family")["Rate"].median().sort_values().index
fig, axes = plt.subplots(2, 4, figsize=(20, 9)); axes = axes.ravel()
for ax, c in zip(axes, BOX):
    v = np.log10(bc[c].replace([np.inf, -np.inf], np.nan).clip(lower=0) + 1)
    sns.boxplot(x=v, y=bc["attack_family"], order=order, ax=ax, fliersize=1, color="#5B6B8C")
    ax.set_title(c); ax.set_xlabel("log10(1 + value)"); ax.set_ylabel("")
plt.suptitle("Key variables by attack family (stratified sample, up to 400 rows per label, log scale)", y=1.01)
plt.tight_layout(); plt.savefig(FIG_DIR / "IoT23_boxplots_by_family.png", dpi=150, bbox_inches="tight"); plt.show()

**What the box plots show.** The families occupy clearly different ranges — denial-of-service and Mirai floods have the highest `Rate` and the shortest `IAT`, benign and web traffic the largest and most varied packets (`AVG`, `Std`, `Max`), ICMP/GRE floods sit at `Header_Length` = 0.
A value that is an outlier for the whole dataset is often *typical* for one family: removing outliers globally would delete attacks.

### Where the anomalies are (% of each label's rows, all rows)

In [ ]:
an = P["anom"].reindex(labels.index).fillna(0)
an_pct = (100 * an.div(labels, axis=0)).round(1)
an_pct.insert(0, "rows", labels)
an.to_csv(TABLE_DIR / "IoT23_phase2_anomalies_by_label.csv")
an_pct

**Interpretation — not every strange value is an error:**
* `Header_Length = 0` covers ≈ 89 % of `DDOS-ICMP_FLOOD` and 54–60 % of the Mirai GRE floods. ICMP and GRE packets have **no TCP/UDP header**, so 0 means *not applicable*: a structural value, keep it.
* `Max > 1514 bytes` (larger than an Ethernet frame) affects ≈ 33 % of benign rows. The packets were captured on the host **before the network card segmented them** (segmentation offload). It is a property of the capture, not an entry error: keep and document.
* `Number < 100` is **100 %** for benign, reconnaissance, spoofing, brute-force and web-attack rows but ≈ 0.1 % for floods. Window size alone almost separates floods from everything else: a **capture artefact** that could act as a shortcut feature (to discuss in phase 5).
* `Time_To_Live = 0` (455 rows) appears in benign, reconnaissance and spoofing windows, which contain non-IP packets (ARP).

In [ ]:
legit = pd.DataFrame([
    ("Rate = inf", "error (division by zero)", int(P["inf"].get("Rate", 0)), "inf -> NaN, drop the rows (3.1)"),
    ("IAT < 0", "error (impossible)", int(P["rules"].get("IAT < 0", 0)), "set to NaN, drop the rows (3.4)"),
    ("Std / Variance = NaN, Number = 1", "structural (spread of one value)", int(P["std_nan_number"].sum()), "impute 0 (3.1)"),
    ("Rate very high (max %.3g pkt/s)" % P["max"]["Rate"], "legitimate: floods", None, "keep; log transform in phase 5"),
    ("Max > 1,514 bytes", "legitimate: capture before segmentation offload", int(P["anom"].get("Max > 1514 bytes", pd.Series(dtype=int)).sum()), "keep, documented"),
    ("Header_Length = 0", "legitimate: ICMP / GRE have no TCP/UDP header", int(P["anom"].get("Header_Length = 0", pd.Series(dtype=int)).sum()), "keep"),
    ("Time_To_Live = 0", "legitimate: non-IP (ARP) windows", int(P["anom"].get("Time_To_Live = 0", pd.Series(dtype=int)).sum()), "keep"),
    ("IAT > 1 hour (max %.0f s)" % P["max"]["IAT"], "doubtful but possible (idle device)", int(P["anom"].get("IAT > 1 hour", pd.Series(dtype=int)).sum()), "keep, flagged in the report"),
], columns=["value", "verdict", "rows (all data)", "planned treatment"])
legit

## 2.5 Data consistency check
### Duplicate records (overall and per class, all rows, by hashing every row)

In [ ]:
dups = pd.Series({
    "exact duplicate rows (same features and label)": P["dup_rows"],
    "rows whose features already appeared (any label)": P["dup_features"],
    "feature vectors that carry more than one label": P["conflict_vectors"],
    "rows inside those conflicting groups": int(P["conflict_rows_by_label"].sum()),
})
display(pd.DataFrame({"count": dups, "% of rows": (100 * dups / R).round(2)}))
by = pd.DataFrame({"rows": labels, "exact duplicates": P["dup_rows_by_label"], "in conflict groups": P["conflict_rows_by_label"]}).fillna(0).astype(int)
by["% duplicated"] = (100 * by["exact duplicates"] / by["rows"]).round(1); by["% in conflict"] = (100 * by["in conflict groups"] / by["rows"]).round(1)
by.to_csv(TABLE_DIR / "IoT23_phase2_duplicates_by_label.csv")
by.sort_values("rows", ascending=False)

**Interpretation.**
* **About half of all rows are exact duplicates.** They come mostly from floods: a flood repeats the same packets, so many 100-packet windows are identical (e.g. ≈ 70 % of `DDOS-ICMP_FLOOD`). After `clean()` the dataset will be roughly half its size.
* **Conflicting labels:** **610,472** distinct feature vectors appear with **more than one label**, covering ≈ 13.3 M rows (29.5 %). They are almost all **DoS vs DDoS of the same flood type** (UDP, TCP, SYN, and the SYN-based Synonymous-IP flood). At window level, one attacker and many attackers produce identical traffic summaries. `clean()` does not remove these rows (they are not exact duplicates once the label is included). This is important for **RQ2** and needs a team decision (see decision log).

### Unique identifiers
The lab checks that `customer_id` is unique. CICIoT2023 has **no identifier at all** (no flow ID, IP, port, device or timestamp), so a record can only be identified by its full set of values —
which is exactly how duplicates were found above. In phase 3 a record is identified by its source file and line number until duplicates are removed.

### Formatting consistency

In [ ]:
fmt = pd.DataFrame({"raw name": first.columns})
fmt["has spaces"] = fmt["raw name"].str.contains(" ")
fmt["leading/trailing spaces"] = fmt["raw name"] != fmt["raw name"].str.strip()
print("column names with spaces:", fmt.loc[fmt["has spaces"], "raw name"].tolist())
print("column names with leading/trailing spaces:", int(fmt["leading/trailing spaces"].sum()))
print("label case:", "all UPPERCASE" if all(l == l.upper() for l in labels.index) else "mixed", "| same spelling in every file:", P["layouts"] == 1)

### Logical checks — shared `validate` (sample) and CICIoT2023-specific rules (all rows)

In [ ]:
display(validate(P["uniform"]))

In [ ]:
rules = P["rules"].reindex(list(RULES)).fillna(0).astype(int).rename("violating rows").to_frame()   # rules with 0 violations included
rules["% of rows"] = (100 * rules["violating rows"] / R).round(5)
rules.to_csv(TABLE_DIR / "IoT23_phase2_rules.csv")
rules

**Interpretation.**
* Every range rule holds (no Min > Max, TTL within 0–255, shares within 0–1, counts ≤ `Number`, `Header_Length` ≤ 60), apart from 2 negative `IAT`.
* The redundancy rules are violated only by the truncated and single-packet rows. On every complete row: **`Tot size` = `AVG`, `LLC` = `IPv`, `Variance` = `Std²`, `Tot sum` = `AVG` × `Number`, and each `*_count` = share × `Number`**. These 8 columns add no information.
* `Protocol Type` always agrees with the `TCP`/`UDP`/`ICMP` shares (0 violations): the two encode the same information.

### Other findings: constant and near-constant columns

In [ ]:
u = P["uniform"].drop(columns=[LABEL_COL])
top_share = u.apply(lambda s: s.value_counts(normalize=True, dropna=False).iloc[0] * 100).round(3)
const = pd.DataFrame({"most frequent value": u.mode().iloc[0], "% of rows with it (sample)": top_share, "distinct values (sample)": u.nunique()})
const.sort_values("% of rows with it (sample)", ascending=False).head(14)

**Interpretation.** No column is strictly constant, but `Telnet`, `SMTP`, `IRC`, `IGMP`, `DHCP`, `SSH`, `ece_flag_number` and `cwr_flag_number` are 0 in more than 99 % of rows: **near-constant**, candidates for removal in feature selection (phase 5).

## 2.6 Class distribution

In [ ]:
labels = P["labels"]; labels = labels[labels.index.notna()].sort_values(ascending=False).astype(int)
dist = map_labels(pd.DataFrame({LABEL_COL: labels.index}), LABEL_COL, SOURCE)      # stops with an error if a label is not in TAXONOMY
dist["rows"] = labels.values
dist["% of rows"] = (100 * dist["rows"] / R).round(4)
dist = dist.rename(columns={LABEL_COL: "original_label"})[["original_label", "attack_label", "attack_family", "rows", "% of rows"]]
dist.to_csv(TABLE_DIR / "IoT23_phase2_labels.csv", index=False)

fig, ax = plt.subplots(1, 2, figsize=(16, 9), gridspec_kw={"width_ratios": [2.2, 1]})
pal = dict(zip(sorted(dist["attack_family"].unique()), sns.color_palette("tab10")))
ax[0].barh(dist["attack_label"][::-1], dist["rows"][::-1], color=[pal[f] for f in dist["attack_family"][::-1]])
ax[0].set_xscale("log"); ax[0].set_xlabel("rows (log scale)"); ax[0].set_title(f"CICIoT2023: rows per attack label ({R/1e6:.1f} M rows)")
fam = dist.groupby("attack_family")["rows"].sum().sort_values()
ax[1].barh(fam.index, fam.values, color=[pal[f] for f in fam.index]); ax[1].set_xscale("log"); ax[1].set_title("Rows per attack family")
plt.tight_layout(); plt.savefig(FIG_DIR / "IoT23_class_distribution.png", dpi=150, bbox_inches="tight"); plt.show()
print(f"{len(dist)} labels -> {dist['attack_family'].nunique()} families | largest/smallest = {labels.iloc[0] / labels.iloc[-1]:,.0f}x "
      f"| benign = {dist.loc[dist.attack_family == BENIGN, '% of rows'].sum():.2f} %")
dist

**Interpretation.** All 34 original labels map to the shared taxonomy without changes (the UPPERCASE spellings are normalised by `map_labels`). The data is dominated by **Denial of service** (16 labels, ≈ 89 % of rows): `DDoS ICMP flood` alone is about 15 %. **Benign is only ≈ 2.3 %**, and the web attacks have a few thousand rows each (`Uploading attack`: 1,196). The largest label is about **5,800 times** the smallest: the sampling cap of 200k rows per label (`sample_per_label`) is essential here.

## 2.7 Summary: problems found → planned solution

## Key variables for the research questions
| Variable | Why it matters |
|---|---|
| `Rate` | Floods are defined by extreme packet rates (RQ1); contains infinite values |
| `Protocol Type` (+ `TCP`/`UDP`/`ICMP` shares) | Protocol mix differs by family: chi-square protocol × family (RQ2) |
| `syn/ack/rst/fin/psh_flag_number` | TCP flag patterns distinguish SYN, ACK, RST-FIN and PSH-ACK floods (RQ1, RQ2) |
| `AVG`, `Std`, `Min`, `Max` | Packet-size profile; benign traffic has larger and more varied packets (RQ1) |
| `IAT` | Timing: slow attacks (SlowLoris) vs floods (RQ1) |
| `Header_Length` | Separates protocols with and without a transport header (ICMP/GRE floods) |
| `Number` | Window size — a capture artefact to watch (possible shortcut) |
| `Label` → `attack_family` | Target; 33 attack labels in 6 attack families + Benign |

## Problems found and planned solutions (no code yet)
| # | Problem found | Columns / rows affected | Planned solution | Rule in `config.py`? / phase-3 step |
|---|---|---|---|---|
| 1 | Release differs from documentation (40 columns, 63 files, 45.0 M rows) | whole dataset | Use the loaded numbers; update `docs/datasets.md` | — (docs) |
| 2 | Label column is `Label`, not `label` | `Label` | Pass `"Label"` to `map_labels` | No → decision log |
| 3 | Truncated lines with many empty cells, 9 rows without label | 7 lines in 7 files | Dropped by `clean()` (NaN rows) | Yes (`clean`) |
| 4 | Integer columns read as float in some files | 8 columns | Fixed by dropping the truncated lines; `downcast()` | Yes (`clean`) |
| 5 | `Std`/`Variance` NaN for single-packet windows | 670 rows (`Number = 1`) | Dropped by `clean()`; documented as structural missingness | Yes (`clean`) |
| 6 | `Rate` infinite | 991 rows | inf → NaN → row dropped by `clean()` | Yes (`clean`) |
| 7 | Negative `IAT` | 2 rows | Set to NaN → dropped | No → decision log |
| 8 | Exact duplicate rows | ≈ 51 % of rows, mostly floods | Dropped by `clean()` (file by file, then again after concatenation) | Yes (`clean`) |
| 9 | Same features with different labels (DoS vs DDoS) | 610,472 vectors, ≈ 13.3 M rows | Team decision needed: keep and report, or drop the groups | No → decision log |
| 10 | Labels in UPPERCASE | 34 labels | Normalised by `map_labels` (no change needed) | Yes (`map_labels`) |
| 11 | Extreme class imbalance | 5,800× between largest and smallest label | `sample_per_label` (cap 200k, rare labels kept in full) | Yes (`sample_per_label`) |
| 12 | Redundant columns (`LLC`, `Tot size`, `Variance`, `Tot sum`, 4 × `*_count`) | 8 columns | Keep in phase 3; remove in feature selection (phase 5) | No → phase 5 |
| 13 | Near-constant columns (`Telnet`, `SMTP`, `IRC`, `IGMP`, `DHCP`, `SSH`, `ece`, `cwr`) | 8 columns | Remove in feature selection (phase 5) | No → phase 5 |
| 14 | `Protocol Type` is a categorical code stored as a number | 1 column | Encode in feature engineering (phase 5) | No → phase 5 |
| 15 | Structural zeros (`Header_Length = 0` for ICMP/GRE), packets > 1514 bytes, TTL = 0 for non-IP windows | ≈ 7.2 M, 0.69 M, 455 rows | Keep: valid measurements, documented | — |
| 16 | `Number` almost separates floods from other classes | `Number` | Flag as capture artefact; decide in phase 5 | No → phase 5 |
| 17 | Extreme outliers (`Rate`, `IAT`, sizes) | several columns | Investigate per class in phase 3; keep legitimate attack values | No → phase 3 |
| 18 | No identifiers or timestamps; files are shuffled | whole dataset | Document as limitation (no time-based split) | — |

**Why CICIoT2023 is not merged with the NetFlow data (task 3.6, short version):** a NetFlow row is one *flow* (one connection) with 53 features from the NetFlow v3 extractor; a CICIoT2023 row is one *window of ≤ 100 packets* with 39 window statistics from a different tool. The features do not measure the same things in the same units, so CICIoT2023 is analysed on its own and linked to the others through the shared attack taxonomy.

---
# Phase 3 — Data preprocessing (apply the plan, with before/after evidence)

Every step follows the plan of 2.7 and the shared contract in `src/config.py`. Decisions not covered by `config.py` are in `docs/decision_log.md`.

**How it runs.** `prepare_file` applies steps 3.1–3.3 to one file (in parallel, 6 files at a time): standardise column names → set impossible values to NaN →
`clean()` (inf → NaN, impute `Std`/`Variance` = 0, drop remaining NaN rows, drop duplicates inside the file) → `map_labels()`.
The 63 cleaned files are then combined, and duplicates **across** files are removed (3.5).

In [ ]:
IMPUTE = {"Std": 0, "Variance": 0}                     # single-packet windows: the spread of one value is 0 (2.2)
INT_COLS = ["ack_count", "syn_count", "fin_count", "rst_count", "Tot_sum", "Min", "Max", "Number"]
LABEL_CATS = pd.CategoricalDtype(sorted({v[0] for v in TAXONOMY.values()}))
FAMILY_CATS = pd.CategoricalDtype([BENIGN] + FAMILIES)
std_name = lambda c: c.strip().replace(" ", "_")

def prepare_file(path):
    df = pd.read_csv(path, low_memory=False)
    df = df.rename(columns=std_name)                                   # 3.3 column names: spaces -> '_'
    st = {"file": path.name, "rows_read": len(df)}
    neg = df["IAT"] < 0                                                 # 3.4 impossible value -> NaN (row dropped by clean)
    st["iat_negative_set_nan"] = int(neg.sum()); df.loc[neg, "IAT"] = np.nan
    st["std_nan_single_packet"] = int((df["Std"].isna() & (df["Number"] == 1)).sum())
    st["rate_inf"] = int(np.isinf(df["Rate"]).sum())
    df, s = clean(df, SOURCE, impute=IMPUTE)                            # 3.1 + 3.5 (inside the file)
    st.update({k: s[k] for k in ("inf_values", "nan_values", "rows_with_nan", "duplicate_rows", "rows_clean")})
    st["missing_report_rows_after"] = len(missing_report(df))          # 0 = no missing / infinite value left in this file
    df = map_labels(df, LABEL_COL, SOURCE)                              # 3.3 labels -> shared taxonomy
    df["attack_label"] = df["attack_label"].astype(LABEL_CATS)
    df["attack_family"] = df["attack_family"].astype(FAMILY_CATS)
    df[LABEL_COL] = df[LABEL_COL].astype("category"); df["source_dataset"] = df["source_dataset"].astype("category")
    return df, st

t0 = time.time()
with mp.get_context("fork").Pool(6) as pool:                            # 6 workers: ~1.5 GB of RAM each
    results = pool.map(prepare_file, FILES)
per_file = pd.DataFrame([st for _, st in results]).set_index("file")
df = pd.concat([d for d, _ in results], ignore_index=True); del results
for c in (LABEL_COL, "source_dataset"): df[c] = df[c].astype("category")
for c in ("attack_label", "attack_family"): df[c] = df[c].cat.remove_unused_categories()   # keep only labels present in IoT23
print(f"prepared {len(FILES)} files in {time.time() - t0:.0f}s -> {len(df):,} rows, {df.shape[1]} columns, "
      f"{df.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")
per_file.sum().to_frame("total over 63 files")

## 3.1 Missing and infinite values — before / after

In [ ]:
mv = pd.DataFrame({
    "before (raw, all rows)": [int(P["nan"].sum()), int(P["inf"].sum()), int(P["std_nan_number"].sum()), int(P["rules"].get("IAT < 0", 0))],
    "treatment": ["truncated lines dropped", "inf -> NaN, rows dropped", "imputed with 0", "set to NaN, rows dropped"],
    "after": [int(df.isna().sum().sum()), int(sum(np.isinf(df[c]).sum() for c in df.select_dtypes("float").columns)), 0, int((df["IAT"] < 0).sum())],
}, index=["missing cells", "infinite cells", "Std/Variance NaN with Number = 1", "negative IAT"])
display(mv)
print("files where missing_report still finds a missing or infinite value after cleaning:", int((per_file["missing_report_rows_after"] > 0).sum()), "of", len(per_file))
print("windows with a single packet kept with Std = Variance = 0:", int(((df["Number"] == 1) & (df["Std"] == 0)).sum()))

**Result.** No missing or infinite value is left. The 670 single-packet windows are kept with `Std` = `Variance` = 0 instead of being lost;
16 broken records, 991 rows with infinite `Rate` and the 2 rows with negative `IAT` were dropped (≈ 0.002 % of the data).

## 3.2 Data type conversion (verified)

In [ ]:
before_types = df.dtypes.astype(str).value_counts()
whole = {c: bool(np.all(np.mod(df[c].to_numpy(), 1) == 0)) for c in INT_COLS}
assert all(whole.values()), f"non-integer values found: {whole}"
for c in INT_COLS:
    df[c] = pd.to_numeric(df[c].astype("int64"), downcast="integer")
df["Protocol_Type"] = df["Protocol_Type"].astype("int8").astype("category")
after_types = df.dtypes.astype(str).value_counts()
print("integer columns verified whole-numbered:", whole)
print("Protocol_Type categories:", list(df["Protocol_Type"].cat.categories))
pd.DataFrame({"before": before_types, "after": after_types}).fillna(0).astype(int)

**Result.** The 8 count/size columns are integers again in every row (they were read as float only because of the truncated lines), `Protocol_Type` is a category,
and the label columns are categories. There is no timestamp column to convert. Encoding (one-hot) is left to feature engineering, after the train/test split.

## 3.3 Standardised column names and labels

In [ ]:
renamed = {c: std_name(c) for c in first.columns if c != std_name(c)}
print("columns renamed:", renamed)
lab_map = (df.groupby([LABEL_COL, "attack_label", "attack_family"], observed=True).size().rename("rows").reset_index()
             .sort_values("rows", ascending=False))
print(f"{lab_map[LABEL_COL].nunique()} original labels -> {lab_map['attack_label'].nunique()} attack labels -> "
      f"{lab_map['attack_family'].nunique()} families (incl. Benign); every label mapped: {df['attack_label'].notna().all()}")
lab_map.head(12)

## 3.4 Outlier treatment — removed vs kept
Only **impossible** values are removed; extreme but possible values are the attacks themselves and are kept (2.4). Counts below are on the cleaned data.

In [ ]:
treat = pd.DataFrame([
    ("IAT < 0", "error", per_file["iat_negative_set_nan"].sum(), "removed (3.1)"),
    ("Rate = inf", "error", per_file["rate_inf"].sum(), "removed (3.1)"),
    ("Rate >= 1,000,000 pkt/s", "legitimate: floods", int((df["Rate"] >= 1e6).sum()), "kept"),
    ("Max > 1,514 bytes", "legitimate: segmentation offload", int((df["Max"] > 1514).sum()), "kept"),
    ("Header_Length = 0", "legitimate: ICMP / GRE", int((df["Header_Length"] == 0).sum()), "kept"),
    ("Time_To_Live = 0", "legitimate: non-IP windows", int((df["Time_To_Live"] == 0).sum()), "kept"),
    ("IAT > 1 hour", "doubtful but possible", int((df["IAT"] > 3600).sum()), "kept, flagged"),
], columns=["value", "verdict", "rows", "decision"])
treat.to_csv(TABLE_DIR / "IoT23_phase3_outliers.csv", index=False)
treat

## 3.5 Duplicates and identifier columns
`clean()` already removed duplicates **inside** each file. The same window can also appear in two different files, so duplicates are removed once more on the combined data.
There is no identifier column to drop (`ID_COLUMNS` matches none).

In [ ]:
n_before = len(df)
dup_mask = df.duplicated()
cross_file = int(dup_mask.sum())
df = df.loc[~dup_mask].reset_index(drop=True)
FEATS = [c for c in df.columns if c not in (LABEL_COL, "source_dataset", "attack_label", "attack_family", "is_attack")]
h = pd.util.hash_pandas_object(df[FEATS], index=False)
n_lab = df.groupby(h.values, observed=True)["attack_label"].nunique()
df["conflicting_features"] = h.map(n_lab).gt(1).to_numpy()          # same window values under more than one label
dd = pd.Series({
    "duplicates removed inside files (clean)": int(per_file["duplicate_rows"].sum()),
    "duplicates removed across files": cross_file,
    "rows after removing duplicates": len(df),
    "rows whose features also appear with another label": int(df["conflicting_features"].sum()),
    "remaining exact duplicates": int(df.duplicated().sum()),
})
display(dd.to_frame("count"))
conf = df[df["conflicting_features"]].groupby("attack_label", observed=True).size().sort_values(ascending=False)
conf.head(10).to_frame("rows with conflicting features")

**Result.** About half of the raw rows were exact duplicates (mostly flood windows). The rows whose identical features carry two labels — almost all
*DoS vs DDoS of the same flood type* — are **kept** and marked with `conflicting_features = True`, as proposed in the decision log (pending team agreement):
they are a finding for SQ1 (DoS and DDoS cannot be told apart at window level) and can be removed with one filter before any model is trained.

## 3.6 Validation and data quality report

In [ ]:
display(validate(df))
d_orig = df.rename(columns={std_name(c): c for c in first.columns})          # rules use the original column names
rules_after = pd.Series({k: int(np.asarray(pd.Series(f(d_orig)).fillna(False), dtype=bool).sum()) for k, f in RULES.items()}, name="violations after")
del d_orig
pd.DataFrame({"violations before (raw)": P["rules"].reindex(list(RULES)).fillna(0).astype(int), "violations after": rules_after})

In [ ]:
quality = pd.DataFrame({
    "before": [R, first.shape[1], int(P["nan"].sum()), int(P["inf"].sum()), int(P["rules"].get("IAT < 0", 0)), int(P["dup_rows"]), int(labels.size)],
    "after": [len(df), df.shape[1], int(df.isna().sum().sum()), int(mv.loc["infinite cells", "after"]), int((df["IAT"] < 0).sum()),
              int(df.duplicated().sum()), int(df["attack_label"].nunique())],
}, index=["rows", "columns", "missing cells", "infinite cells", "negative IAT", "exact duplicate rows", "labels"])
quality["change"] = quality["after"] - quality["before"]
quality.to_csv(TABLE_DIR / "IoT23_phase3_before_after.csv")
quality

**Result.** `validate` and all CICIoT2023 rules return **0 violations**: the range rules hold, and the redundancy rules (`Tot_size` = `AVG`, `LLC` = `IPv`,
`Variance` = `Std`², `Tot_sum` = `AVG` × `Number`, counts = share × `Number`) now hold on every row — the redundant columns are removed in feature selection (phase 5).
The extra columns are the four added by `map_labels` and `conflicting_features`.

## 3.7 Stratified sampling and save
`sample_per_label` keeps every row of a label with ≤ 200,000 rows and draws 200,000 at random (random_state = 42) from larger labels.

In [ ]:
sampled = sample_per_label(df)
stats = {"source_dataset": SOURCE, "rows_raw": R, "cols_raw": first.shape[1], "id_columns_dropped": "none",
         "inf_values": int(per_file["inf_values"].sum()), "nan_values": int(per_file["nan_values"].sum()),
         "imputed_columns": ", ".join(f"{c}={v}" for c, v in IMPUTE.items()),
         "rows_with_nan": int(per_file["rows_with_nan"].sum()), "iat_negative_removed": int(per_file["iat_negative_set_nan"].sum()),
         "duplicate_rows": int(per_file["duplicate_rows"].sum()) + cross_file, "duplicates_across_files": cross_file,
         "conflicting_feature_rows": int(df["conflicting_features"].sum()),
         "rows_clean": len(df), "cols_clean": df.shape[1], "rows_sampled": len(sampled)}
lab_table = quality_report(stats, df[["attack_label", "attack_family"]], sampled, SOURCE)
path = save_clean(sampled, SOURCE)
print(f"saved {path.relative_to(ROOT)}: {len(sampled):,} rows x {sampled.shape[1]} columns ({path.stat().st_size / 1e6:.0f} MB)")
print("every label still present after sampling:", set(lab_table.index) == set(df["attack_label"].unique()))
lab_table.sort_values("flows_after_cleaning", ascending=False)

In [ ]:
fam = pd.DataFrame({"raw": dist.groupby("attack_family")["rows"].sum(),
                    "after cleaning": df["attack_family"].value_counts(),
                    "after sampling": sampled["attack_family"].value_counts()}).fillna(0).astype(int).sort_values("raw", ascending=False)
fam = fam[fam["raw"] > 0]
fig, ax = plt.subplots(figsize=(11, 4.5))
fam.plot.barh(ax=ax, logx=True, color=["#C9D3E6", "#5B6B8C", "#E8871E"]); ax.invert_yaxis()
ax.set_title("CICIoT2023 rows per family: raw → cleaned → sampled (log scale)"); ax.set_xlabel("rows")
plt.tight_layout(); plt.savefig(FIG_DIR / "IoT23_rows_per_family_phase3.png", dpi=150, bbox_inches="tight"); plt.show()
fam["% of sampled"] = (100 * fam["after sampling"] / fam["after sampling"].sum()).round(2)
fam

## Why CICIoT2023 is kept separate from the NetFlow data (task 3.10)
* **Different unit of observation.** A NetFlow v3 row is one *flow* (one connection between two endpoints, from its first to its last packet). A CICIoT2023 row is one
  *window of up to 100 consecutive packets*, which can mix several connections.
* **Different features.** NetFlow v3 has 53 features (bytes and packets per direction, durations in ms, TCP flag bitmasks, inter-arrival statistics per direction, ports, L7 protocol).
  CICIoT2023 has 39 window statistics (shares of packets with each flag or protocol, mean packet size, header length, rate). Only a few concepts overlap, and none is measured the same way
  (e.g. a TCP flag is a bitmask per flow in NetFlow but a share of packets per window here).
* **Different tool and units.** The NetFlow v3 files were produced by the same extractor for all four datasets; CICIoT2023 comes from CIC's own Python/DPKT pipeline.
* **Consequence.** Stacking the two would put incompatible measurements in the same column. CICIoT2023 is therefore analysed on its own and compared with the NetFlow data
  **at the level of results** (attack family, shared taxonomy): e.g. whether floods have high rates and small packets in both representations (SQ2).